# 第八章：Order Matters - Sequence to Sequence for Sets

**論文資訊**
- 標題：Order Matters: Sequence to Sequence for Sets
- 作者：Oriol Vinyals, Samy Bengio, Manjunath Kudlur
- 機構：Google Brain
- 發表：ICLR 2016
- 連結：https://arxiv.org/abs/1511.06391

本 notebook 使用 PyTorch 實作論文中的「讀取-處理-寫入」架構，用於處理無序集合資料。

In [ ]:
# 跨平台中文字型設定（支援 Colab、VSCode、Antigravity 等本地環境）
import subprocess
import os
import shutil
import platform

system = platform.system()

# 必須在 import matplotlib 之前清除快取
cache_dir = os.path.expanduser('~/.matplotlib')
if os.path.exists(cache_dir):
    for f in os.listdir(cache_dir):
        if f.startswith('fontlist'):
            try:
                os.remove(os.path.join(cache_dir, f))
            except:
                pass

cache_dir2 = os.path.expanduser('~/.cache/matplotlib')
if os.path.exists(cache_dir2):
    shutil.rmtree(cache_dir2, ignore_errors=True)

# Linux/Colab 環境安裝中文字型
if system == 'Linux' or 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    try:
        result = subprocess.run(['fc-list', ':lang=zh'], capture_output=True, text=True)
        if 'Noto Sans CJK' not in result.stdout:
            print("正在安裝中文字型...")
            subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
            subprocess.run(['apt-get', 'install', '-qq', '-y', 'fonts-noto-cjk'], capture_output=True)
            print("中文字型安裝完成，請重新啟動 kernel")
    except:
        pass

print(f"✓ {system} 環境")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# 重建字型快取並設定中文字型
from matplotlib.font_manager import fontManager
fontManager.__init__()

chinese_fonts = [
    'Heiti TC', 'PingFang TC', 'Noto Sans CJK TC',
    'Heiti SC', 'PingFang SC', 'Noto Sans CJK SC', 
    'Microsoft JhengHei', 'Microsoft YaHei',
    'SimHei', 'WenQuanYi Micro Hei', 'Arial Unicode MS',
]

available_fonts = set(f.name for f in fontManager.ttflist)
selected_font = None
for font in chinese_fonts:
    if font in available_fonts:
        selected_font = font
        break

if selected_font:
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = [selected_font] + chinese_fonts
    plt.rcParams['axes.unicode_minus'] = False
    print(f"✓ 使用中文字型: {selected_font}")
else:
    plt.rcParams['font.sans-serif'] = chinese_fonts
    plt.rcParams['axes.unicode_minus'] = False
    print("⚠ 使用預設字型列表")

from typing import Tuple, Optional, List

# 設定中文字型

# 設定隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 裝置設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用裝置: {device}")

## 8.1 排列不變的集合編碼器

集合編碼器的核心要求是**排列不變性**：無論輸入元素的順序如何，都應產生相同的編碼結果。

數學上，函數 $f$ 具有排列不變性，若對於任意排列 $\pi$：

$$f(\{x_1, x_2, \ldots, x_n\}) = f(\{x_{\pi(1)}, x_{\pi(2)}, \ldots, x_{\pi(n)}\})$$

實作策略包括：
- 平均池化：$f(X) = \frac{1}{n} \sum_i \phi(x_i)$
- 總和池化：$f(X) = \sum_i \phi(x_i)$
- 最大池化：$f(X) = \max_i \phi(x_i)$
- 注意力池化：使用可學習的注意力權重

In [ ]:
class SetEncoder(nn.Module):
    """
    排列不變的集合編碼器
    
    透過對各元素嵌入進行池化運算，確保編碼結果與輸入順序無關。
    
    參數:
        input_dim: 輸入元素的維度
        hidden_dim: 隱藏層維度
        pooling: 池化方式 ('mean', 'sum', 'max', 'attention')
    """
    
    def __init__(self, input_dim: int, hidden_dim: int, pooling: str = 'mean'):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.pooling = pooling
        
        # 元素嵌入層
        self.embed = nn.Linear(input_dim, hidden_dim)
        
        # 注意力池化參數
        if pooling == 'attention':
            self.attn_weight = nn.Linear(hidden_dim, 1)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        前向傳遞
        
        參數:
            x: (batch_size, set_size, input_dim) 輸入集合
        
        回傳:
            encoding: (batch_size, hidden_dim) 集合編碼
            element_encodings: (batch_size, set_size, hidden_dim) 各元素嵌入
        """
        # 嵌入每個元素
        element_encodings = torch.tanh(self.embed(x))  # (B, S, H)
        
        # 排列不變的池化運算
        if self.pooling == 'mean':
            encoding = element_encodings.mean(dim=1)
        elif self.pooling == 'sum':
            encoding = element_encodings.sum(dim=1)
        elif self.pooling == 'max':
            encoding, _ = element_encodings.max(dim=1)
        elif self.pooling == 'attention':
            # 可學習的注意力權重
            attn_logits = self.attn_weight(element_encodings)  # (B, S, 1)
            attn_weights = F.softmax(attn_logits, dim=1)
            encoding = (attn_weights * element_encodings).sum(dim=1)
        else:
            raise ValueError(f"不支援的池化方式: {self.pooling}")
        
        return encoding, element_encodings

In [ ]:
# 驗證排列不變性
print("測試排列不變性")
print("=" * 50)

encoder = SetEncoder(input_dim=1, hidden_dim=16, pooling='mean').to(device)

# 建立集合與其排列
set1 = torch.tensor([[[1.0], [2.0], [3.0], [4.0]]]).to(device)
set2 = torch.tensor([[[4.0], [2.0], [1.0], [3.0]]]).to(device)  # 相同元素，不同順序

enc1, _ = encoder(set1)
enc2, _ = encoder(set2)

print(f"集合 1: {set1.squeeze().cpu().numpy()}")
print(f"集合 2: {set2.squeeze().cpu().numpy()}")
print(f"\n編碼差異: {(enc1 - enc2).norm().item():.10f}")
print(f"編碼是否相同: {torch.allclose(enc1, enc2)}")
print("\n✓ 排列不變性驗證通過！")

## 8.2 LSTM 編碼器（順序敏感基線）

作為對照，我們實作一個標準的 LSTM 編碼器。這個編碼器對輸入順序敏感，在處理集合資料時會產生不一致的結果。

In [ ]:
class LSTMEncoder(nn.Module):
    """
    標準 LSTM 編碼器（順序敏感）
    
    作為基線模型，展示順序敏感編碼器在集合任務上的問題。
    """
    
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        前向傳遞
        
        參數:
            x: (batch_size, seq_len, input_dim) 輸入序列
        
        回傳:
            encoding: (batch_size, hidden_dim) 最終隱藏狀態
            all_hidden: (batch_size, seq_len, hidden_dim) 所有隱藏狀態
        """
        all_hidden, (h_n, c_n) = self.lstm(x)
        encoding = h_n.squeeze(0)
        return encoding, all_hidden

In [ ]:
# 驗證 LSTM 的順序敏感性
print("測試 LSTM 順序敏感性")
print("=" * 50)

lstm_encoder = LSTMEncoder(input_dim=1, hidden_dim=16).to(device)

enc1, _ = lstm_encoder(set1)
enc2, _ = lstm_encoder(set2)

print(f"序列 1: {set1.squeeze().cpu().numpy()}")
print(f"序列 2: {set2.squeeze().cpu().numpy()}")
print(f"\n編碼差異: {(enc1 - enc2).norm().item():.6f}")
print(f"編碼是否相同: {torch.allclose(enc1, enc2)}")
print("\n✓ LSTM 是順序敏感的（符合預期）")

## 8.3 基於內容的注意力機制

解碼器使用注意力機制來關注輸入集合中的相關元素。

注意力分數計算：
$$\text{score}(h_t, e_i) = v^T \tanh(W_1 h_t + W_2 e_i)$$

注意力權重：
$$\alpha_t = \text{softmax}(\text{scores})$$

上下文向量：
$$c_t = \sum_i \alpha_{t,i} \cdot e_i$$

In [ ]:
class ContentAttention(nn.Module):
    """
    基於內容的注意力機制
    
    讓解碼器能夠關注輸入集合中的相關元素。
    """
    
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # 注意力參數
        self.W_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, query: torch.Tensor, keys: torch.Tensor, 
                mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        計算注意力權重與上下文向量
        
        參數:
            query: (batch_size, hidden_dim) 解碼器隱藏狀態
            keys: (batch_size, set_size, hidden_dim) 編碼器元素嵌入
            mask: (batch_size, set_size) 可選的遮罩
        
        回傳:
            context: (batch_size, hidden_dim) 上下文向量
            weights: (batch_size, set_size) 注意力權重
        """
        batch_size, set_size, _ = keys.shape
        
        # 轉換 query 和 keys
        q = self.W_query(query).unsqueeze(1)  # (B, 1, H)
        k = self.W_key(keys)  # (B, S, H)
        
        # 計算注意力分數
        scores = self.v(torch.tanh(q + k)).squeeze(-1)  # (B, S)
        
        # 應用遮罩（如有）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax 得到注意力權重
        weights = F.softmax(scores, dim=-1)  # (B, S)
        
        # 計算上下文向量
        context = torch.bmm(weights.unsqueeze(1), keys).squeeze(1)  # (B, H)
        
        return context, weights

In [ ]:
# 測試注意力機制
print("測試注意力機制")
print("=" * 50)

attention = ContentAttention(hidden_dim=16).to(device)

# 模擬解碼器狀態和編碼器輸出
query = torch.randn(2, 16).to(device)  # batch_size=2
keys = torch.randn(2, 5, 16).to(device)  # 5 個集合元素

context, weights = attention(query, keys)

print(f"Query 形狀: {query.shape}")
print(f"Keys 形狀: {keys.shape}")
print(f"Context 形狀: {context.shape}")
print(f"\n注意力權重（第一個樣本）: {weights[0].detach().cpu().numpy()}")
print(f"權重總和: {weights[0].sum().item():.6f}（應為 1.0）")
print("\n✓ 注意力機制運作正確")

## 8.4 帶注意力的 LSTM 解碼器

解碼器在每個時間步對輸入集合進行注意力，逐一生成輸出元素。

解碼過程：
1. 使用當前隱藏狀態計算對輸入集合的注意力
2. 獲取上下文向量
3. 將上下文與前一個輸出結合
4. 更新 LSTM 狀態
5. 預測下一個輸出元素

In [ ]:
class AttentionDecoder(nn.Module):
    """
    帶注意力機制的 LSTM 解碼器
    
    透過對集合元素進行注意力來生成輸出序列。
    """
    
    def __init__(self, output_dim: int, hidden_dim: int):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        
        # LSTM 輸入：[prev_output, context]
        self.lstm = nn.LSTMCell(output_dim + hidden_dim, hidden_dim)
        
        # 輸出投影
        self.output_proj = nn.Linear(hidden_dim, output_dim)
        
        # 注意力機制
        self.attention = ContentAttention(hidden_dim)
    
    def forward(self, encoder_outputs: torch.Tensor, target_length: int,
                start_token: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        生成完整輸出序列
        
        參數:
            encoder_outputs: (batch_size, set_size, hidden_dim) 編碼的集合元素
            target_length: 輸出序列長度
            start_token: (batch_size, output_dim) 初始輸入
        
        回傳:
            outputs: (batch_size, target_length, output_dim) 預測輸出
            all_attn_weights: (batch_size, target_length, set_size) 注意力權重
        """
        batch_size = encoder_outputs.shape[0]
        device = encoder_outputs.device
        
        # 初始化
        if start_token is None:
            start_token = torch.zeros(batch_size, self.output_dim, device=device)
        
        # 用編碼器輸出的平均值初始化解碼器狀態
        h = encoder_outputs.mean(dim=1)
        c = torch.zeros_like(h)
        
        outputs = []
        all_attn_weights = []
        prev_output = start_token
        
        for t in range(target_length):
            # 1. 計算注意力
            context, attn_weights = self.attention(h, encoder_outputs)
            all_attn_weights.append(attn_weights)
            
            # 2. 結合前一個輸出和上下文
            lstm_input = torch.cat([prev_output, context], dim=-1)
            
            # 3. LSTM 步驟
            h, c = self.lstm(lstm_input, (h, c))
            
            # 4. 預測輸出
            output = self.output_proj(h)
            outputs.append(output)
            prev_output = output
        
        outputs = torch.stack(outputs, dim=1)  # (B, T, D)
        all_attn_weights = torch.stack(all_attn_weights, dim=1)  # (B, T, S)
        
        return outputs, all_attn_weights

## 8.5 完整的 Set2Seq 模型

整合所有元件成為完整的「讀取-處理-寫入」架構：

1. **讀取（Read）**：排列不變編碼器處理輸入集合
2. **處理（Process）**：注意力機制關注集合元素
3. **寫入（Write）**：LSTM 解碼器生成有序輸出

In [ ]:
class Set2Seq(nn.Module):
    """
    完整的 Set2Seq 模型
    
    實作讀取-處理-寫入架構：
    - 排列不變集合編碼器
    - 基於內容的注意力機制
    - LSTM 解碼器
    """
    
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int,
                 pooling: str = 'mean'):
        super().__init__()
        self.encoder = SetEncoder(input_dim, hidden_dim, pooling=pooling)
        self.decoder = AttentionDecoder(output_dim, hidden_dim)
    
    def forward(self, input_set: torch.Tensor, target_length: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        前向傳遞：集合 → 序列
        
        參數:
            input_set: (batch_size, set_size, input_dim) 輸入集合
            target_length: 輸出序列長度
        
        回傳:
            outputs: (batch_size, target_length, output_dim) 預測序列
            attn_weights: (batch_size, target_length, set_size) 注意力權重
        """
        # 編碼集合（排列不變）
        _, element_encodings = self.encoder(input_set)
        
        # 解碼為序列（帶注意力）
        outputs, attn_weights = self.decoder(element_encodings, target_length)
        
        return outputs, attn_weights


class Seq2Seq(nn.Module):
    """
    基線模型：順序敏感的 Seq2Seq
    
    使用 LSTM 編碼器，在處理集合時會因順序不同產生不同結果。
    """
    
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int):
        super().__init__()
        self.encoder = LSTMEncoder(input_dim, hidden_dim)
        self.decoder = AttentionDecoder(output_dim, hidden_dim)
    
    def forward(self, input_seq: torch.Tensor, target_length: int) -> Tuple[torch.Tensor, torch.Tensor]:
        # 編碼序列（順序敏感）
        _, all_hidden = self.encoder(input_seq)
        
        # 解碼
        outputs, attn_weights = self.decoder(all_hidden, target_length)
        
        return outputs, attn_weights


print("✓ Set2Seq 和 Seq2Seq 模型已建立")

## 8.6 排序任務資料集

使用數字排序作為測試任務：
- 輸入：無序數字集合，如 {3, 1, 4, 2}
- 輸出：排序後的序列，如 [1, 2, 3, 4]

這個任務能有效測試排列不變性，因為無論輸入順序如何，正確答案都是相同的排序結果。

In [ ]:
class SortingDataset(Dataset):
    """
    排序任務資料集
    
    生成隨機數字集合及其排序後的結果。
    """
    
    def __init__(self, num_samples: int = 1000, set_size: int = 5, 
                 value_range: int = 10, permute: bool = False):
        self.num_samples = num_samples
        self.set_size = set_size
        self.value_range = value_range
        self.permute = permute
        
        # 生成資料
        self.data = np.random.randint(0, value_range, size=(num_samples, set_size, 1)).astype(np.float32)
        self.targets = np.sort(self.data, axis=1)
        
        # 標準化
        self.data = self.data / value_range
        self.targets = self.targets / value_range
    
    def __len__(self) -> int:
        return self.num_samples
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.data[idx])
        y = torch.from_numpy(self.targets[idx])
        
        # 是否隨機排列輸入
        if self.permute:
            perm = torch.randperm(self.set_size)
            x = x[perm]
        
        return x, y


# 建立資料集
train_dataset = SortingDataset(num_samples=5000, set_size=5, value_range=10, permute=True)
test_dataset = SortingDataset(num_samples=500, set_size=5, value_range=10, permute=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("排序任務資料集")
print("=" * 50)
print(f"訓練樣本數: {len(train_dataset)}")
print(f"測試樣本數: {len(test_dataset)}")
print(f"集合大小: {train_dataset.set_size}")

# 範例
x, y = train_dataset[0]
print(f"\n範例:")
print(f"  輸入集合: {(x.numpy().flatten() * 10).astype(int)}")
print(f"  排序輸出: {(y.numpy().flatten() * 10).astype(int)}")

## 8.7 訓練迴圈

訓練 Set2Seq 模型學習排序任務。

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader, optimizer: optim.Optimizer,
                criterion: nn.Module, device: torch.device) -> float:
    """訓練一個 epoch"""
    model.train()
    total_loss = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # 前向傳遞
        outputs, _ = model(x, target_length=y.shape[1])
        
        # 計算損失
        loss = criterion(outputs, y)
        
        # 反向傳播
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module,
             device: torch.device) -> Tuple[float, float]:
    """評估模型"""
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            
            outputs, _ = model(x, target_length=y.shape[1])
            loss = criterion(outputs, y)
            total_loss += loss.item()
            
            # 計算排序準確率
            pred_sorted = outputs.argsort(dim=1)
            true_sorted = y.argsort(dim=1)
            correct = (pred_sorted == true_sorted).all(dim=1).all(dim=1).sum().item()
            total_correct += correct
            total_samples += x.shape[0]
    
    avg_loss = total_loss / len(loader)
    accuracy = total_correct / total_samples
    
    return avg_loss, accuracy

In [ ]:
# 訓練 Set2Seq 模型
print("訓練 Set2Seq 模型")
print("=" * 60)

model = Set2Seq(input_dim=1, output_dim=1, hidden_dim=64, pooling='mean').to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

num_epochs = 30
train_losses = []
test_losses = []
test_accs = []

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | 訓練損失: {train_loss:.6f} | "
              f"測試損失: {test_loss:.6f} | 準確率: {test_acc:.2%}")

print("\n✓ 訓練完成")

## 8.8 訓練基線模型（Seq2Seq）

In [ ]:
# 訓練 Seq2Seq 基線模型
print("訓練 Seq2Seq 基線模型")
print("=" * 60)

baseline_model = Seq2Seq(input_dim=1, output_dim=1, hidden_dim=64).to(device)
baseline_optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)

baseline_train_losses = []
baseline_test_losses = []
baseline_test_accs = []

for epoch in range(num_epochs):
    train_loss = train_epoch(baseline_model, train_loader, baseline_optimizer, criterion, device)
    test_loss, test_acc = evaluate(baseline_model, test_loader, criterion, device)
    
    baseline_train_losses.append(train_loss)
    baseline_test_losses.append(test_loss)
    baseline_test_accs.append(test_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | 訓練損失: {train_loss:.6f} | "
              f"測試損失: {test_loss:.6f} | 準確率: {test_acc:.2%}")

print("\n✓ 基線模型訓練完成")

## 8.9 視覺化結果

In [ ]:
# 視覺化訓練過程
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. 訓練損失
ax = axes[0]
ax.plot(train_losses, label='Set2Seq', linewidth=2)
ax.plot(baseline_train_losses, label='Seq2Seq', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('訓練損失', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. 測試損失
ax = axes[1]
ax.plot(test_losses, label='Set2Seq', linewidth=2)
ax.plot(baseline_test_losses, label='Seq2Seq', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('測試損失', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 3. 測試準確率
ax = axes[2]
ax.plot(test_accs, label='Set2Seq', linewidth=2)
ax.plot(baseline_test_accs, label='Seq2Seq', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('準確率', fontsize=12)
ax.set_title('測試準確率', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n最終結果:")
print(f"  Set2Seq - 測試損失: {test_losses[-1]:.6f}, 準確率: {test_accs[-1]:.2%}")
print(f"  Seq2Seq - 測試損失: {baseline_test_losses[-1]:.6f}, 準確率: {baseline_test_accs[-1]:.2%}")

## 8.10 排列不變性驗證

In [ ]:
def test_permutation_invariance(model: nn.Module, x: torch.Tensor, 
                                 num_perms: int = 10) -> Tuple[List[float], float]:
    """
    測試模型在多個排列下的輸出一致性
    
    回傳:
        losses: 每個排列的損失
        variance: 輸出的變異數
    """
    model.eval()
    outputs_list = []
    
    with torch.no_grad():
        for _ in range(num_perms):
            # 隨機排列
            perm = torch.randperm(x.shape[1])
            x_perm = x[:, perm, :]
            
            output, _ = model(x_perm, target_length=x.shape[1])
            outputs_list.append(output)
    
    # 計算輸出之間的變異數
    outputs = torch.stack(outputs_list)  # (num_perms, B, T, D)
    variance = outputs.var(dim=0).mean().item()
    
    # 計算每個排列的損失（與平均輸出的差異）
    mean_output = outputs.mean(dim=0)
    losses = [(o - mean_output).pow(2).mean().item() for o in outputs_list]
    
    return losses, variance


# 測試排列不變性
print("排列不變性測試")
print("=" * 60)

# 取一批測試資料
test_x, test_y = next(iter(test_loader))
test_x = test_x[:10].to(device)  # 取 10 個樣本

# 測試 Set2Seq
set2seq_losses, set2seq_var = test_permutation_invariance(model, test_x, num_perms=20)
print(f"Set2Seq:")
print(f"  平均損失: {np.mean(set2seq_losses):.8f}")
print(f"  損失標準差: {np.std(set2seq_losses):.8f}")
print(f"  輸出變異數: {set2seq_var:.8f}")

# 測試 Seq2Seq
seq2seq_losses, seq2seq_var = test_permutation_invariance(baseline_model, test_x, num_perms=20)
print(f"\nSeq2Seq:")
print(f"  平均損失: {np.mean(seq2seq_losses):.8f}")
print(f"  損失標準差: {np.std(seq2seq_losses):.8f}")
print(f"  輸出變異數: {seq2seq_var:.8f}")

print(f"\n結論:")
print(f"  Set2Seq 輸出變異數 / Seq2Seq 輸出變異數 = {set2seq_var / seq2seq_var:.4f}")
print(f"  Set2Seq 對輸入順序更加不敏感！")

In [ ]:
# 視覺化排列不變性
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. 損失分布比較
ax = axes[0]
ax.bar([0], [np.mean(set2seq_losses)], yerr=[np.std(set2seq_losses)], 
       width=0.4, label='Set2Seq', alpha=0.7, capsize=5)
ax.bar([1], [np.mean(seq2seq_losses)], yerr=[np.std(seq2seq_losses)], 
       width=0.4, label='Seq2Seq', alpha=0.7, capsize=5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Set2Seq', 'Seq2Seq'])
ax.set_ylabel('排列間的損失變異', fontsize=12)
ax.set_title('排列不變性比較', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 2. 多次排列的損失分布
ax = axes[1]
x_pos = np.arange(len(set2seq_losses))
width = 0.35
ax.bar(x_pos - width/2, set2seq_losses, width, label='Set2Seq', alpha=0.7)
ax.bar(x_pos + width/2, seq2seq_losses, width, label='Seq2Seq', alpha=0.7)
ax.set_xlabel('排列編號', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('各排列的損失（應該接近）', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('permutation_invariance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.11 注意力權重視覺化

In [ ]:
# 視覺化注意力權重
model.eval()

# 取一個樣本
sample_x, sample_y = test_dataset[0]
sample_x = sample_x.unsqueeze(0).to(device)
sample_y = sample_y.unsqueeze(0).to(device)

with torch.no_grad():
    outputs, attn_weights = model(sample_x, target_length=sample_y.shape[1])

# 轉換為 numpy
input_values = (sample_x.cpu().numpy().squeeze() * 10).astype(int)
output_values = outputs.cpu().numpy().squeeze() * 10
target_values = (sample_y.cpu().numpy().squeeze() * 10).astype(int)
attn = attn_weights.cpu().numpy().squeeze()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. 輸入 vs 輸出
ax = axes[0]
x_pos = np.arange(len(input_values))
ax.plot(x_pos, input_values, 'o-', label='輸入（無序）', markersize=12, linewidth=2)
ax.plot(x_pos, target_values, 's-', label='目標（已排序）', markersize=12, linewidth=2, alpha=0.7)
ax.plot(x_pos, output_values, '^--', label='預測', markersize=12, linewidth=2, alpha=0.7)
ax.set_xlabel('位置', fontsize=12)
ax.set_ylabel('值', fontsize=12)
ax.set_title('排序任務：輸入 vs 輸出', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. 注意力熱圖
ax = axes[1]
im = ax.imshow(attn, aspect='auto', cmap='YlOrRd')
ax.set_xlabel('輸入位置', fontsize=12)
ax.set_ylabel('輸出時間步', fontsize=12)
ax.set_title('注意力權重', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='權重')

# 標註輸入值
ax.set_xticks(range(len(input_values)))
ax.set_xticklabels(input_values)

plt.tight_layout()
plt.savefig('attention_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n輸入集合: {input_values}")
print(f"預測輸出: {output_values.round(1)}")
print(f"目標輸出: {target_values}")

## 8.12 消融研究：池化策略比較

In [ ]:
# 消融研究：比較不同池化策略
print("消融研究：池化策略比較")
print("=" * 60)

pooling_methods = ['mean', 'sum', 'max', 'attention']
pooling_results = {}

for pooling in pooling_methods:
    print(f"\n訓練 {pooling.upper()} 池化模型...")
    
    # 建立模型
    ablation_model = Set2Seq(input_dim=1, output_dim=1, hidden_dim=64, pooling=pooling).to(device)
    ablation_optimizer = optim.Adam(ablation_model.parameters(), lr=0.001)
    
    # 訓練（較少 epoch 以節省時間）
    ablation_losses = []
    for epoch in range(15):
        train_loss = train_epoch(ablation_model, train_loader, ablation_optimizer, criterion, device)
        ablation_losses.append(train_loss)
    
    # 評估
    test_loss, test_acc = evaluate(ablation_model, test_loader, criterion, device)
    _, perm_var = test_permutation_invariance(ablation_model, test_x, num_perms=10)
    
    pooling_results[pooling] = {
        'test_loss': test_loss,
        'test_acc': test_acc,
        'perm_variance': perm_var,
        'train_losses': ablation_losses
    }
    
    print(f"  測試損失: {test_loss:.6f} | 準確率: {test_acc:.2%} | 排列變異: {perm_var:.8f}")

In [ ]:
# 視覺化消融研究結果
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

methods = list(pooling_results.keys())
test_losses_ablation = [pooling_results[m]['test_loss'] for m in methods]
test_accs_ablation = [pooling_results[m]['test_acc'] for m in methods]
perm_vars = [pooling_results[m]['perm_variance'] for m in methods]

colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid']

# 1. 測試損失
ax = axes[0]
ax.bar(methods, test_losses_ablation, color=colors, alpha=0.7)
ax.set_xlabel('池化方法', fontsize=12)
ax.set_ylabel('測試損失', fontsize=12)
ax.set_title('各池化策略的測試損失', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 2. 測試準確率
ax = axes[1]
ax.bar(methods, test_accs_ablation, color=colors, alpha=0.7)
ax.set_xlabel('池化方法', fontsize=12)
ax.set_ylabel('測試準確率', fontsize=12)
ax.set_title('各池化策略的準確率', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 3. 排列不變性（變異數越低越好）
ax = axes[2]
ax.bar(methods, perm_vars, color=colors, alpha=0.7)
ax.set_xlabel('池化方法', fontsize=12)
ax.set_ylabel('排列變異數（越低越好）', fontsize=12)
ax.set_title('排列不變性程度', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

# 找出最佳池化方法
best_method = min(pooling_results.keys(), key=lambda k: pooling_results[k]['test_loss'])
print(f"\n最佳池化方法: {best_method.upper()}")
print(f"測試損失: {pooling_results[best_method]['test_loss']:.6f}")
print(f"準確率: {pooling_results[best_method]['test_acc']:.2%}")

## 8.13 更大集合的泛化測試

In [ ]:
# 測試在更大集合上的泛化能力
print("泛化測試：不同集合大小")
print("=" * 60)

set_sizes = [3, 5, 7, 10]
generalization_results = []

for size in set_sizes:
    # 建立測試資料集
    gen_dataset = SortingDataset(num_samples=200, set_size=size, value_range=10, permute=True)
    gen_loader = DataLoader(gen_dataset, batch_size=32, shuffle=False)
    
    # 評估
    test_loss, test_acc = evaluate(model, gen_loader, criterion, device)
    generalization_results.append({'size': size, 'loss': test_loss, 'acc': test_acc})
    
    print(f"集合大小 {size}: 損失 = {test_loss:.6f}, 準確率 = {test_acc:.2%}")

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sizes = [r['size'] for r in generalization_results]
losses_gen = [r['loss'] for r in generalization_results]
accs_gen = [r['acc'] for r in generalization_results]

ax = axes[0]
ax.plot(sizes, losses_gen, 'o-', linewidth=2, markersize=10)
ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='訓練集合大小')
ax.set_xlabel('集合大小', fontsize=12)
ax.set_ylabel('測試損失', fontsize=12)
ax.set_title('不同集合大小的損失', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(sizes, accs_gen, 'o-', linewidth=2, markersize=10, color='green')
ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='訓練集合大小')
ax.set_xlabel('集合大小', fontsize=12)
ax.set_ylabel('準確率', fontsize=12)
ax.set_title('不同集合大小的準確率', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 8.14 模型摘要與連結

### 實作總結

本章實作了「讀取-處理-寫入」架構：

1. **排列不變的集合編碼器**：透過池化運算確保輸入順序不影響編碼結果
2. **基於內容的注意力機制**：讓解碼器能關注輸入集合中的相關元素
3. **LSTM 解碼器**：生成有序的輸出序列

### 關鍵發現

- Set2Seq 相比傳統 Seq2Seq 在處理集合資料時更加穩定
- 不同池化策略有各自的優缺點，平均池化通常是良好的預設選擇
- 模型在一定程度上能泛化到不同大小的集合

### 相關論文連結

- 論文連結：https://arxiv.org/abs/1511.06391

In [ ]:
# 顯示論文 QR code
from IPython.display import Image, display
import os

qr_path = 'paper_qrcode.png'
if os.path.exists(qr_path):
    display(Image(filename=qr_path, width=200))
    print("\n掃描 QR code 閱讀原始論文")
else:
    print("論文連結: https://arxiv.org/abs/1511.06391")

In [ ]:
print("=" * 70)
print("第八章：Order Matters - Sequence to Sequence for Sets")
print("=" * 70)
print("""
✅ 實作完成

本章展示了如何使用序列到序列模型處理無序集合資料。

主要成就：

1. 架構元件
   • 排列不變的集合編碼器（支援多種池化策略）
   • 基於內容的注意力機制
   • 帶注意力的 LSTM 解碼器
   • Seq2Seq 基線模型（用於對照）

2. 實驗驗證
   • 排序任務（經典集合問題）
   • 排列不變性測試
   • Set2Seq vs Seq2Seq 比較
   • 池化策略消融研究
   • 不同集合大小的泛化測試

3. 關鍵洞見
   • 排列不變性對集合任務至關重要
   • 注意力機制提供可解釋性
   • 池化策略的選擇影響模型性能

相關章節連結：
• 第六章（Pointer Networks）：可變輸出長度、基於注意力的選擇
• 第十四章（Bahdanau 注意力）：原始注意力機制
• 第十三章（Transformer）：自注意力機制
""")
print("=" * 70)
print("🎓 第八章實作完成！")
print("=" * 70)